# Crop Augmentation Ablation

Ce notebook reprend le script `crop_augmentation_ablation.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Teste les augmentations qui rendent les modeles crop plus robustes en streaming.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Repeated-split crop augmentation ablation for attention and blouse/PPE.
- Commande de reproduction referencee : crop augmentation ablation.
- Artefacts controles : Repeated-split attention/blouse crop augmentation ablation exists. (`runs/exp_067_crop_augmentation_ablation/metrics/crop_augmentation_ablation_summary.csv`).
- Run par defaut : `runs/exp_067_crop_augmentation_ablation`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "crop_augmentation_ablation.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
import random
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from crop_cnn_experiments import TARGETS, make_model, predict
from crop_cnn_stability_experiments import parent_combo_split
from ml_pipeline import ROOT, safe_auc, write_json
from sequence_experiments import append_report, make_run_dir


POLICIES = {
    "none": {
        "summary": "resize only; no stochastic training augmentation",
        "train_only": True,
    },
    "photometric_only": {
        "summary": "resize plus color jitter and mild blur; no spatial perturbation",
        "train_only": True,
    },
    "geometry_only": {
        "summary": "random resized crop and small rotation; no color jitter or flip",
        "train_only": True,
    },
    "standard_no_flip": {
        "summary": "random resized crop, color jitter, small rotation, mild blur; no horizontal flip",
        "train_only": True,
    },
    "standard_with_flip": {
        "summary": "current crop-CNN training recipe with random resized crop, p=0.25 horizontal flip, color jitter, rotation, blur",
        "train_only": True,
    },
}


## Fonction `set_seed`

Cette cellule definit `set_seed`. Elle prepare une partie du script.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


## Fonction `resolve`

Cette cellule definit `resolve`. Elle prepare une partie du script.

In [ ]:
def resolve(path):
    path = Path(path)
    return path if path.is_absolute() else ROOT / path


## Fonction `image_path`

Cette cellule definit `image_path`. Elle prepare une partie du script.

In [ ]:
def image_path(run_dir, row_path):
    path = Path(str(row_path))
    return path if path.is_absolute() else Path(run_dir) / path


## Fonction `make_transform`

Cette cellule definit `make_transform`. Elle prepare une partie du script.

In [ ]:
def make_transform(policy, train, image_size):
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    if not train:
        return transforms.Compose([transforms.Resize((image_size, image_size)), transforms.ToTensor(), normalize])
    if policy == "none":
        return transforms.Compose([transforms.Resize((image_size, image_size)), transforms.ToTensor(), normalize])
    if policy == "photometric_only":
        return transforms.Compose(
            [
                transforms.Resize((image_size, image_size)),
                transforms.RandomApply([transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.18, hue=0.03)], p=0.75),
                transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.20),
                transforms.ToTensor(),
                normalize,
            ]
        )
    if policy == "geometry_only":
        return transforms.Compose(
            [
                transforms.RandomResizedCrop(image_size, scale=(0.85, 1.0), ratio=(0.90, 1.10)),
                transforms.RandomRotation(degrees=5),
                transforms.ToTensor(),
                normalize,
            ]
        )
    if policy == "standard_no_flip":
        return transforms.Compose(
            [
                transforms.RandomResizedCrop(image_size, scale=(0.80, 1.0), ratio=(0.80, 1.25)),
                transforms.RandomApply([transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.18, hue=0.03)], p=0.75),
                transforms.RandomRotation(degrees=6),
                transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.20),
                transforms.ToTensor(),
                normalize,
            ]
        )
    if policy == "standard_with_flip":
        return transforms.Compose(
            [
                transforms.RandomResizedCrop(image_size, scale=(0.80, 1.0), ratio=(0.80, 1.25)),
                transforms.RandomHorizontalFlip(p=0.25),
                transforms.RandomApply([transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.18, hue=0.03)], p=0.75),
                transforms.RandomRotation(degrees=6),
                transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.20),
                transforms.ToTensor(),
                normalize,
            ]
        )
    raise ValueError(policy)


## Classe `PolicyCropDataset`

Cette cellule definit `PolicyCropDataset`. Elle prepare une partie du script.

In [ ]:
class PolicyCropDataset(Dataset):
    def __init__(self, run_dir, index, target, policy, train=False, image_size=224):
        self.run_dir = Path(run_dir)
        self.index = index.reset_index(drop=True)
        self.target = target
        self.tf = make_transform(policy, train, image_size)

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        row = self.index.iloc[idx]
        image = Image.open(image_path(self.run_dir, row["path"])).convert("RGB")
        x = self.tf(image)
        y = torch.tensor(float(row[f"{self.target}_label"]), dtype=torch.float32)
        return x, y


## Fonction `threshold_metrics`

Cette cellule definit `threshold_metrics`. Elle prepare une partie du script.

In [ ]:
def threshold_metrics(y, p):
    best = None
    for threshold in np.arange(0.05, 1.0, 0.05):
        pred = (p >= threshold).astype(int)
        item = {
            "threshold": float(threshold),
            "f1": float(f1_score(y, pred, zero_division=0)),
            "accuracy": float(accuracy_score(y, pred)),
            "balanced_accuracy": float(balanced_accuracy_score(y, pred)) if len(np.unique(y)) > 1 else np.nan,
            "confusion_matrix": confusion_matrix(y, pred, labels=[0, 1]).tolist(),
        }
        if best is None or item["f1"] > best["f1"]:
            best = item
    return best


## Fonction `evaluate_predictions`

Cette cellule definit `evaluate_predictions`. Elle prepare une partie du script.

In [ ]:
def evaluate_predictions(pred, target):
    rows = []
    for level in ["crop", "video"]:
        if level == "video":
            eval_df = (
                pred.groupby(["split", "video_id"], as_index=False)
                .agg(label=(f"{target}_label", "max"), risk=("risk", "mean"))
                .copy()
            )
        else:
            eval_df = pred.rename(columns={f"{target}_label": "label"}).copy()
        for split, group in eval_df.groupby("split"):
            y = group["label"].astype(int).to_numpy()
            p = group["risk"].astype(float).to_numpy()
            best = threshold_metrics(y, p)
            rows.append(
                {
                    "level": level,
                    "split": split,
                    "n": int(len(group)),
                    "positive": int(y.sum()),
                    "average_precision": safe_auc(average_precision_score, y, p),
                    "roc_auc": safe_auc(roc_auc_score, y, p),
                    **best,
                }
            )
    return rows


## Fonction `train_one`

Cette cellule definit `train_one`. Elle prepare une partie du script.

In [ ]:
def train_one(run_dir, index, target, architecture, policy, args, device):
    train_df = index[index["split"].eq("train")].copy()
    val_df = index[index["split"].eq("val")].copy()
    train_ds = PolicyCropDataset(run_dir, train_df, target, policy, train=True, image_size=args.image_size)
    val_ds = PolicyCropDataset(run_dir, val_df, target, policy, train=False, image_size=args.image_size)
    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, num_workers=0)
    model = make_model(architecture, pretrained=not args.no_pretrained).to(device)
    y_train = train_df[f"{target}_label"].to_numpy()
    positives = max(1, int(y_train.sum()))
    negatives = max(1, int(len(y_train) - positives))
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([negatives / positives], dtype=torch.float32, device=device))
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=args.lr, weight_decay=args.weight_decay)
    best = {"ap": -1.0, "state": None, "epoch": 0}
    history = []
    patience_left = args.patience
    start = time.perf_counter()
    for epoch in range(1, args.epochs + 1):
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb).view(-1), yb)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
        val_probs, val_y = predict(model, val_loader, device)
        val_ap = float(safe_auc(average_precision_score, val_y.astype(int), val_probs) or 0.0)
        history.append(
            {
                "target": target,
                "architecture": architecture,
                "augmentation_policy": policy,
                "epoch": epoch,
                "train_loss": float(np.mean(losses)),
                "val_ap": val_ap,
            }
        )
        if val_ap > best["ap"] + 1e-5:
            best = {"ap": val_ap, "state": {k: v.detach().cpu() for k, v in model.state_dict().items()}, "epoch": epoch}
            patience_left = args.patience
        else:
            patience_left -= 1
        if patience_left <= 0:
            break
    if best["state"] is not None:
        model.load_state_dict(best["state"])
    train_time = time.perf_counter() - start
    model_path = run_dir / "models" / f"{target}_{architecture}_{policy}.pt"
    torch.save(
        {
            "target": target,
            "architecture": architecture,
            "augmentation_policy": policy,
            "state_dict": model.state_dict(),
            "best_epoch": best["epoch"],
        },
        model_path,
    )
    return model, history, train_time, model_path.stat().st_size


## Fonction `summarize`

Cette cellule definit `summarize`. Elle prepare une partie du script.

In [ ]:
def summarize(metrics):
    metric_cols = ["average_precision", "roc_auc", "f1", "balanced_accuracy"]
    group_cols = ["target", "architecture", "augmentation_policy", "level", "split"]
    rows = []
    for keys, group in metrics.groupby(group_cols):
        row = dict(zip(group_cols, keys))
        row["n_repeats"] = int(group["repeat_seed"].nunique())
        row["n_mean"] = float(group["n"].mean())
        row["positive_mean"] = float(group["positive"].mean())
        for col in metric_cols:
            row[f"{col}_mean"] = float(group[col].mean())
            row[f"{col}_std"] = float(group[col].std(ddof=0))
            row[f"{col}_min"] = float(group[col].min())
            row[f"{col}_max"] = float(group[col].max())
        rows.append(row)
    return pd.DataFrame(rows)


## Fonction `policy_for_target`

Cette cellule definit `policy_for_target`. Elle prepare une partie du script.

In [ ]:
def policy_for_target(target, args):
    mapping = {}
    for item in args.target_architectures:
        key, value = item.split("=", 1)
        mapping[key] = value
    return mapping[target]


## Fonction `preview_image`

Cette cellule definit `preview_image`. Elle prepare une partie du script.

In [ ]:
def preview_image(image, policy, image_size):
    img = image.resize((image_size, image_size))
    if policy == "none":
        return img
    if policy == "photometric_only":
        img = ImageEnhance.Brightness(img).enhance(1.18)
        img = ImageEnhance.Contrast(img).enhance(1.25)
        return img.filter(ImageFilter.GaussianBlur(radius=0.6))
    if policy == "geometry_only":
        w, h = img.size
        crop = img.crop((int(w * 0.06), int(h * 0.04), int(w * 0.96), int(h * 0.96)))
        return crop.resize((image_size, image_size)).rotate(4, resample=Image.Resampling.BILINEAR)
    if policy == "standard_no_flip":
        img = preview_image(image, "geometry_only", image_size)
        img = ImageEnhance.Brightness(img).enhance(1.16)
        return ImageEnhance.Contrast(img).enhance(1.20)
    if policy == "standard_with_flip":
        img = preview_image(image, "standard_no_flip", image_size)
        return img.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
    raise ValueError(policy)


## Fonction `write_augmentation_audit`

Cette cellule definit `write_augmentation_audit`. Elle prepare une partie du script.

In [ ]:
def write_augmentation_audit(run_dir, index, args):
    rows = []
    train = index[index["split"].eq("train")].head(args.preview_samples).copy()
    preview_dir = run_dir / "error_review" / "crop_augmentation_examples"
    preview_dir.mkdir(parents=True, exist_ok=True)
    tile = 144
    label_h = 34
    sheet = Image.new("RGB", (tile * len(args.augmentation_policies), (tile + label_h) * len(train)), "white")
    draw = ImageDraw.Draw(sheet)
    for row_i, (_, row) in enumerate(train.iterrows()):
        src = Image.open(image_path(run_dir, row["path"])).convert("RGB")
        for col_i, policy in enumerate(args.augmentation_policies):
            preview = preview_image(src, policy, tile)
            x = col_i * tile
            y = row_i * (tile + label_h)
            sheet.paste(preview, (x, y + label_h))
            draw.text((x + 4, y + 3), policy[:18], fill=(0, 0, 0))
            draw.text((x + 4, y + 17), f"split=train frame={int(row['frame'])}", fill=(0, 0, 0))
            rows.append(
                {
                    "video_id": row["video_id"],
                    "frame": int(row["frame"]),
                    "split": row["split"],
                    "augmentation_policy": policy,
                    "source_path": row["path"],
                    "applied_to_validation_or_test": False,
                    "policy_summary": POLICIES[policy]["summary"],
                }
            )
    sheet.save(preview_dir / "crop_augmentation_policy_contact_sheet.jpg", quality=92)
    manifest_rows = []
    for policy in args.augmentation_policies:
        if policy == "none":
            continue
        for _, row in index[index["split"].eq("train")].iterrows():
            manifest_rows.append(
                {
                    "video_id": row["video_id"],
                    "frame": int(row["frame"]),
                    "split": row["split"],
                    "augmentation_policy": policy,
                    "source_path": row["path"],
                    "applied_to_validation_or_test": False,
                    "policy_summary": POLICIES[policy]["summary"],
                }
            )
    pd.DataFrame(manifest_rows).to_csv(run_dir / "features" / "crop_augmentation_manifest.csv", index=False)
    pd.DataFrame(rows).to_csv(run_dir / "features" / "crop_augmentation_preview_manifest.csv", index=False)
    audit = {
        "policies": {policy: POLICIES[policy]["summary"] for policy in args.augmentation_policies},
        "train_source_crops": int((index["split"] == "train").sum()),
        "val_source_crops": int((index["split"] == "val").sum()),
        "test_source_crops": int((index["split"] == "test").sum()),
        "manifest_rows": int(len(manifest_rows)),
        "validation_or_test_augmented_rows": 0,
        "contact_sheet": str(preview_dir / "crop_augmentation_policy_contact_sheet.jpg"),
    }
    write_json(run_dir / "metrics" / "crop_augmentation_audit.json", audit)


## Fonction `write_summary`

Cette cellule definit `write_summary`. Elle prepare une partie du script.

In [ ]:
def write_summary(run_dir, summary, args):
    lines = ["# Crop Augmentation Ablation", ""]
    lines.append("This audit isolates training-time image augmentation policies for attention and blouse/PPE crop classifiers. Parent videos are split first; derived crops inherit the parent split; validation/test transforms are resize-only.")
    lines.append("")
    lines.append("## Policies")
    lines.append("")
    for policy in args.augmentation_policies:
        lines.append(f"- `{policy}`: {POLICIES[policy]['summary']}")
    lines.append("")
    lines.append("## Held-Out Test Summary")
    lines.append("")
    lines.append("| target | architecture | level | rank | policy | repeats | AP mean | AP std | ROC AUC | F1 | balanced acc |")
    lines.append("|---|---|---|---:|---|---:|---:|---:|---:|---:|---:|")
    test = summary[summary["split"].eq("test")].copy()
    for (target, architecture, level), group in test.sort_values(
        ["target", "architecture", "level", "average_precision_mean"],
        ascending=[True, True, True, False],
    ).groupby(["target", "architecture", "level"]):
        for rank, (_, row) in enumerate(group.iterrows(), start=1):
            lines.append(
                f"| {target} | {architecture} | {level} | {rank} | {row['augmentation_policy']} | {int(row['n_repeats'])} | "
                f"{row['average_precision_mean']:.3f} | {row['average_precision_std']:.3f} | {row['roc_auc_mean']:.3f} | "
                f"{row['f1_mean']:.3f} | {row['balanced_accuracy_mean']:.3f} |"
            )
    lines.append("")
    lines.append("## Interpretation Guardrails")
    lines.append("")
    lines.append("- Validation/test crops are not augmented; augmentation is train-only.")
    lines.append("- Policy choice should be made from validation summaries and then reported against test, not selected directly from test.")
    lines.append("- Horizontal flip is treated as a hypothesis because the machine/camera geometry is fixed and left-right semantics can matter.")
    (run_dir / "crop_augmentation_ablation_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    source_run = resolve(args.crop_run)
    run_dir = make_run_dir(args.run_name)
    source_index = pd.read_csv(source_run / "features" / "crop_cnn_index.csv")
    index = source_index.copy()
    index["path"] = index["path"].apply(lambda p: str(source_run / p))
    write_json(
        run_dir / "config.json",
        {
            "crop_run": str(source_run),
            "seeds": args.seeds,
            "augmentation_policies": args.augmentation_policies,
            "target_architectures": args.target_architectures,
            "epochs": args.epochs,
            "patience": args.patience,
            "split_policy": "parent video split stratified by attention/blouse combination; crops inherit parent split",
            "validation_test_augmentation": "none; resize-only evaluation transform",
        },
    )
    write_augmentation_audit(run_dir, index, args)
    device = torch.device("cuda" if torch.cuda.is_available() and args.device == "auto" else args.device)
    all_metrics = []
    all_history = []
    split_rows = []
    for seed in args.seeds:
        set_seed(seed)
        split = parent_combo_split(index, seed)
        split_index = index.copy()
        split_index["split"] = split_index["video_id"].map(split)
        split_index.to_csv(run_dir / "features" / f"crop_aug_split_seed_{seed}.csv", index=False)
        for split_name, group in split_index.groupby("split"):
            split_rows.append(
                {
                    "repeat_seed": seed,
                    "split": split_name,
                    "videos": int(group["video_id"].nunique()),
                    "crops": int(len(group)),
                    "attention_positive_crops": int(group["attention_label"].sum()),
                    "blouse_positive_crops": int(group["blouse_label"].sum()),
                }
            )
        for target in TARGETS:
            architecture = policy_for_target(target, args)
            for policy in args.augmentation_policies:
                print(f"training seed={seed} target={target} arch={architecture} policy={policy} device={device}")
                set_seed(seed)
                model, history, train_time_s, model_size = train_one(run_dir, split_index, target, architecture, policy, args, device)
                model_path = run_dir / "models" / f"{target}_{architecture}_{policy}.pt"
                renamed = run_dir / "models" / f"{target}_seed{seed}_{architecture}_{policy}.pt"
                if model_path.exists():
                    model_path.replace(renamed)
                for row in history:
                    row["repeat_seed"] = seed
                all_history.extend(history)
                eval_loader = DataLoader(
                    PolicyCropDataset(run_dir, split_index, target, policy, train=False, image_size=args.image_size),
                    batch_size=args.batch_size,
                    shuffle=False,
                    num_workers=0,
                )
                probs, _ = predict(model, eval_loader, device)
                pred = split_index[["video_id", "split", "frame", "time_s", f"{target}_label"]].copy()
                pred["target"] = target
                pred["architecture"] = architecture
                pred["augmentation_policy"] = policy
                pred["repeat_seed"] = seed
                pred["risk"] = probs
                pred.to_csv(run_dir / "features" / f"predictions_{target}_seed{seed}_{architecture}_{policy}.csv", index=False)
                for metric in evaluate_predictions(pred, target):
                    metric.update(
                        {
                            "target": target,
                            "architecture": architecture,
                            "augmentation_policy": policy,
                            "repeat_seed": seed,
                            "train_time_s": train_time_s,
                            "model_size_bytes": model_size,
                        }
                    )
                    all_metrics.append(metric)
                pd.DataFrame(all_metrics).to_csv(run_dir / "metrics" / "crop_augmentation_ablation_metrics.csv", index=False)
                pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "crop_augmentation_training_history.csv", index=False)
    metrics = pd.DataFrame(all_metrics)
    summary = summarize(metrics)
    metrics.to_csv(run_dir / "metrics" / "crop_augmentation_ablation_metrics.csv", index=False)
    summary.to_csv(run_dir / "metrics" / "crop_augmentation_ablation_summary.csv", index=False)
    pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "crop_augmentation_training_history.csv", index=False)
    pd.DataFrame(split_rows).to_csv(run_dir / "metrics" / "crop_augmentation_split_counts.csv", index=False)
    write_summary(run_dir, summary, args)
    append_report(run_dir, "Crop Augmentation Ablation", f"- Summary: `{run_dir / 'crop_augmentation_ablation_summary.md'}`")
    print(run_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Repeated-split crop augmentation ablation for attention and blouse/PPE.")
    parser.add_argument("--crop-run", default="runs/exp_013_crop_cnn_catalogue")
    parser.add_argument("--run-name", default="exp_067_crop_augmentation_ablation")
    parser.add_argument("--augmentation-policies", nargs="+", default=list(POLICIES.keys()))
    parser.add_argument("--target-architectures", nargs="+", default=["attention=small_cnn", "blouse=resnet18"])
    parser.add_argument("--seeds", nargs="+", type=int, default=[101, 202, 303, 404, 505])
    parser.add_argument("--image-size", type=int, default=224)
    parser.add_argument("--epochs", type=int, default=6)
    parser.add_argument("--patience", type=int, default=2)
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--lr", type=float, default=3e-4)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--preview-samples", type=int, default=8)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--no-pretrained", action="store_true")
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_067_crop_augmentation_ablation_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["crop_augmentation_ablation.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
